In [ ]:
list_of_pairs = [[[0,2,0],[0,1,1]], [[0,2,0],[0,2,2]], [[0,2,0],[0,3,0]], [[0,2,0],[0,3,2]], [[0,2,0],[0,3,1]], [[0,2,0],[0,3,-1]], 
                 [[0,2,0],[0,4,1]], [[0,2,0],[0,4,0]], [[0,2,0],[0,4,-1]], [[0,2,0],[0,4,-2]]] + \
                [[[-2,3,-1], [-2,3,-2]], [[-2,3,-1], [-2,3,-3]], [[1, 3, 2], [1, 3, 3]]] + \
                [[[-1, 4, -2], [-1, 4, -3]], [[-2, 3, -3], [-2, 4, -4]], [[1, 4, 1], [1, 4, 2]], [[2, 4, 2], [2, 4, 3]], [[2, 2, 2], [2, 4, 3]], [[2, 4, 3], [2, 4, 4]]] + \
                [[[-2, 3, -3], [-2, 2, -2]], [[-2, 3, -2], [-2, 2, -1]], [[2, 4, 2], [2, 2, 1]], [[1, 2, 2], [1, 3, 2]]] + \
                [[[-2, 3, -3], [-2, 1, -1]], [[-2, 3, -1], [-2, 1, 0]]]
                
                
                
list_inv_bussed_pairs =[[[0,3,-1], [-1,3,-1]], [[0,3,-1], [-2,3,-1]], [[0,3,2], [1,3,2]]] + [[[0,4,1], [1, 4, 1]], [[1, 4, 2], [2, 4, 2]], [[0, 4, -2], [-1, 4, -2]]] + [[[0,2, 0], [-2,2, 0]], [[0, 2, 0], [-1, 2, 0]], [[0, 2, 0], [2, 2, 0]]]

In [18]:
len(list_of_pairs)

25

In [3]:
list_of_pairs = [[[0, 1, 1], [0, 2, 0]], [[0, 2, 2], [0, 2, 0]], [[0, 3, 0], [0, 2, 0]], [[0, 3, 2], [0, 2, 0]], [[0, 3, 1], [0, 2, 0]], [[0, 3, -1], [0, 2, 0]], [[0, 4, 1], [0, 2, 0]], [[0, 4, 0], [0, 2, 0]], 
                 [[0, 4, -1], [0, 2, 0]], [[0, 4, -2], [0, 2, 0]]] +\
                [[[-2, 3, -2], [-2, 3, -1]], [[-2, 3, -3], [-2, 3, -1]], [[1, 3, 3], [1, 3, 2]], [[-1, 4, -3], [-1, 4, -2]], [[-2, 4, -4], [-2, 3, -3]], [[1, 4, 2], [1, 4, 1]], [[2, 4, 3], [2, 4, 2]], [[2, 4, 3], [2, 2, 2]], [[2, 4, 4], [2, 4, 3]], [[-2, 2, -2], [-2, 3, -3]], [[-2, 2, -1], [-2, 3, -2]], [[2, 2, 1], [2, 4, 2]], [[1, 3, 2], [1, 2, 2]], [[-2, 1, -1], [-2, 3, -3]], [[-2, 1, 0], [-2, 3, -1]]]

In [4]:
list_inv_bussed_pairs = [[[-1, 3, -1], [0, 3, -1]], [[-2, 3, -1], [0, 3, -1]], [[1, 3, 2], [0, 3, 2]], [[1, 4, 1], [0, 4, 1]], [[2, 4, 2], [1, 4, 2]], [[-1, 4, -2], [0, 4, -2]], [[-2, 2, 0], [0, 2, 0]], [[-1, 2, 0], [0, 2, 0]], [[2, 2, 0], [0, 2, 0]]]

In [25]:
print(len(list_inv_bussed_pairs))

9


In [58]:
import numpy as np
import matplotlib.pyplot as plt
import re
import os
from datetime import datetime
from scipy.optimize import curve_fit, fsolve
import io
def compute_phases_from_line_fit(
    pulse_train,
    fractions,
    pi_t=(23.76, 36.54, 106.33, 32.755, 39.168),
    B0=0.2289,
    A60=-0.2791,
    phi60=1.0002,
    A180=-0.0774,
    phi180=-0.2650,
    param_file="line_signal_fit_params.txt",
    t_ref="start",  # "mid", "start", or "end"
    transition_strengths_path=r"Z:\Lab Data\Phase_and_freq_correction_180Hz\Transition_strengths_4p209.txt",
    sensitivities_path=r"Z:\Lab Data\Phase_and_freq_correction_180Hz\sensitivities_4p209.txt",
    integral_scale=1.1
):
    """
    Single-function version of your entire pipeline.

    Computes per-pulse phase corrections using:
        A(t) = t*B(t) - ∫_0^t B(t') dt'
    with B(t) = B_rel_G(t) (Gauss), so A(t) is in G*s.

    Phase per pulse i:
        phi_i = 2π * 1e6 * sens_i * A(t_i)

    Returns:
        phases_per_pulse (list[float]): same length as pulse_train
    """

    # -----------------------------
    # Helpers (nested)
    # -----------------------------
    def load_line_fit_parameters(filename):
        B0_fit = None
        B0_err = None
        harmonic_lines = []
        with open(filename, "r") as f:
            for line in f:
                stripped = line.strip()
                if not stripped or stripped.startswith("#"):
                    continue
                if B0_fit is None:
                    vals = np.fromstring(stripped, sep=" ")
                    B0_fit, B0_err = vals[0], vals[1]
                else:
                    harmonic_lines.append(stripped)
        harmonics = np.loadtxt(io.StringIO("\n".join(harmonic_lines)))
        if harmonics.ndim == 1:
            harmonics = harmonics[None, :]
        freqs = harmonics[:, 0]
        A_vals = harmonics[:, 1]
        A_errs = harmonics[:, 2]
        phi_vals = harmonics[:, 3]
        phi_errs = harmonics[:, 4]
        return B0_fit, B0_err, freqs, A_vals, A_errs, phi_vals, phi_errs

    def _get_line_params(param_file, B0, A60, phi60, A180, phi180):
        if param_file is not None:
            B0_fit, B0_err, freqs, A_vals, A_errs, phi_vals, phi_errs = load_line_fit_parameters(param_file)
            return B0_fit, freqs, A_vals, phi_vals
        freqs = np.array([60.0, 180.0], dtype=float)
        A_vals = np.array([A60, A180], dtype=float)
        phi_vals = np.array([phi60, phi180], dtype=float)
        return B0, freqs, A_vals, phi_vals

    def B_line_mG(t, B0, A60, phi60, A180, phi180, param_file):
        B0_use, freqs, A_vals, phi_vals = _get_line_params(param_file, B0, A60, phi60, A180, phi180)
        t_arr = np.asarray(t, dtype=float)
        y = np.full_like(t_arr, B0_use, dtype=float)
        for f, A, phi in zip(freqs, A_vals, phi_vals):
            y = y + A * np.cos(2.0 * np.pi * f * t_arr + phi)
        return y

    def B_rel_G(t, B0, A60, phi60, A180, phi180, param_file):
        return (
            B_line_mG(t, B0, A60, phi60, A180, phi180, param_file=param_file)
            - B_line_mG(0.0, B0, A60, phi60, A180, phi180, param_file=param_file)
        ) * 1e-3

    def primitive_B_rel_G(t, B0, A60, phi60, A180, phi180, param_file):
        B0_use, freqs, A_vals, phi_vals = _get_line_params(param_file, B0, A60, phi60, A180, phi180)
        t_arr = np.asarray(t, dtype=float)
        t_arr_1d = np.atleast_1d(t_arr)
        omega = 2.0 * np.pi * freqs

        B_line_0 = B0_use + np.sum(A_vals * np.cos(phi_vals))
        const_term = (B0_use - B_line_0) * t_arr_1d
        sin_terms = (A_vals / omega)[:, None] * np.sin(omega[:, None] * t_arr_1d[None, :] + phi_vals[:, None])

        F_mG = const_term + np.sum(sin_terms, axis=0)
        F_Gs = 1e-3 * F_mG
        if np.isscalar(t):
            return float(F_Gs[0])
        return F_Gs

    def integral_B_rel_G(t_start, t_end, B0, A60, phi60, A180, phi180, param_file):
        F_end = primitive_B_rel_G(t_end, B0, A60, phi60, A180, phi180, param_file=param_file)
        F_start = primitive_B_rel_G(t_start, B0, A60, phi60, A180, phi180, param_file=param_file)
        return F_end - F_start

    def compute_pi_times(pi_t):
        transition_strengths = np.loadtxt(transition_strengths_path, delimiter=",")
        transition_strengths[transition_strengths == 0] = np.nan
        strengths = np.array(
            [
                transition_strengths[22, 1],
                transition_strengths[14, 0],
                transition_strengths[5, 2],
                transition_strengths[16, 4],
                transition_strengths[15, 4],
            ]
        )
        factors = np.array(pi_t, dtype=float) * strengths

        Fs = [1, 2, 3, 4]
        row_labels = [[i, i - j] for i in Fs for j in range(2 * i + 1)]
        col_labels = [-2, -1, 0, 1, 2]

        pi_times = np.zeros((24, 5), dtype=float)
        for i in range(transition_strengths.shape[0]):
            for j in range(transition_strengths.shape[1]):
                if not np.isnan(transition_strengths[i, j]):
                    delta_m = (row_labels[i][1] - col_labels[j]) + 2
                    pi_times[i, j] = factors[delta_m] / transition_strengths[i, j]
        return pi_times

    def get_pi_times_from_matrix(transitions, matrix):
        Fs = [1, 2, 3, 4]
        states = []
        for i in Fs:
            for j in range(2 * i + 1):
                mF = i - j
                states.append([i, mF])
        row_labels = states
        col_labels = [-2, -1, 0, 1, 2]

        out = []
        for transition in transitions:
            row_label = [transition[1], transition[2]]
            col_label = transition[0]
            row_index = next((k for k, label in enumerate(row_labels) if label == row_label), None)
            col_index = col_labels.index(col_label) if col_label in col_labels else None
            if row_index is not None and col_index is not None:
                out.append(matrix[row_index, col_index])
            else:
                out.append(np.nan)
        return out

    def get_pulse_schedule(rabi_freqs, fractions):
        if len(rabi_freqs) != len(fractions):
            raise ValueError(
                f"rabi_freqs {len(rabi_freqs)} and fractions {len(fractions)} must have the same length."
            )
        times = []
        t_current = 0.0
        for Omega, frac in zip(rabi_freqs, fractions):
            if not 0 <= frac <= 1:
                raise ValueError(f"Fraction must be between 0 and 1, got {frac}.")
            theta = 2.0 * np.arcsin(np.sqrt(frac))
            t_pulse = theta / Omega if Omega > 0 else 0.0
            t_start = t_current
            t_end = t_current + t_pulse
            times.append((t_start, t_end))
            t_current = t_end
        return times

    # -----------------------------
    # Main logic (formerly top-level)
    # -----------------------------
    pulse_train = [tuple(tr) for tr in pulse_train]

    # Build pulse schedule from pi-times
    pi_times_matrix = compute_pi_times(pi_t)
    pi_times_train = get_pi_times_from_matrix(pulse_train, pi_times_matrix)

    # Free placeholder: effectively no pulse
    for i, tr in enumerate(pulse_train):
        if tr == (0, 0, 0):
            pi_times_train[i] = 1e6

    rabi_frequencies_list = np.pi / np.array(pi_times_train, dtype=float)
    schedule_us = get_pulse_schedule(rabi_frequencies_list, fractions)
    pulses_sec = [(start * 1e-6, end * 1e-6) for (start, end) in schedule_us]

    # Sensitivities
    sens_matrix = np.loadtxt(sensitivities_path, delimiter=",")
    sens_list = get_pi_times_from_matrix(pulse_train, sens_matrix)
    for i, tr in enumerate(pulse_train):
        if tr == (0, 0, 0):
            sens_list[i] = 0.0
    sens_array = np.array(sens_list, dtype=float)
    sens_array_mod = [-np.abs(sens_array[0]), np.abs(sens_array[1]), np.abs(sens_array[2]), np.abs(sens_array[3]), -np.abs(sens_array[4])]
    # Reference time per pulse
    t_refs = np.zeros(len(pulse_train), dtype=float)
    for i, (t_start, t_end) in enumerate(pulses_sec):
        if t_ref == "start":
            t_refs[i] = t_start
        elif t_ref == "end":
            t_refs[i] = t_end
        else:  # "mid"
            t_refs[i] = 0.5 * (t_start + t_end)
    # print(t_refs)
    # Compute phases
    phases_per_pulse = np.zeros(len(pulse_train), dtype=float)
    detunings = np.zeros(len(pulse_train), dtype=float)
    sys_detunings = [integral_scale, -integral_scale, 0, -integral_scale, integral_scale]
    for i, tr in enumerate(pulse_train):
        if tr == (0, 0, 0) or sens_array[i] == 0.0:
            phases_per_pulse[i] = 0.0
            continue

        ti = float(t_refs[i])
        # print(ti)
        Bt = B_rel_G(ti, B0, A60, phi60, A180, phi180, param_file=param_file)
        I0t = integral_B_rel_G(0.0, ti, B0, A60, phi60, A180, phi180, param_file=param_file)
        area_Gs = 0*ti * Bt - I0t #+ ti*integral_scale

        phases_per_pulse[i] = 2.0 * np.pi * 1e6 * sens_array_mod[i] * area_Gs + 2*np.pi*sys_detunings[i] * ti
        detunings[i] = Bt * sens_array_mod[i]
    return phases_per_pulse.tolist(), sens_array_mod

In [66]:
import numpy as np

def compute_theory_phase_mod_and_sens(
    max_waittime_us,
    target_transition,
    step_us=100,
    pi_t=np.array([21.48, 33.79, 108.27, 26.98, 35.26]),
    integral_scale=1.1,
):
    """
    Builds pulse_train from target_transition, computes theoretical phase_mod vs waittime.

    Args:
        max_waittime_us (float): maximum wait time in microseconds (e.g. max(wait_time_list))
        target_transition (list): [[m,F,mF],[m,F,mF]] from Cell 1
        step_us (int): step size in microseconds for the scan (default 100)
        pi_t: pi-times array passed through to compute_phases_from_line_fit
        integral_scale (float): multiplier applied to the integral term in Cell 4

    Returns:
        theory_phases (np.ndarray): phase_mod = mod(accm_phase, 2π)/π
        sens_list (list[float]): sensitivity list returned by compute_phases_from_line_fit
    """
    # Build pulse train: [t1, t2, idle, t2, t1]
    t1 = target_transition[0]
    t2 = target_transition[1]
    pulse_train = [t1, t2, [0, 0, 0], t2, t1]

    accm_phase = []
    sens_list = None  # store once (should be constant for fixed pulse_train)

    # Use max_waittime_us instead of hard-coded 2200
    # step_us = max_waittime_us/100
    list_of_times = np.arange(0, max_waittime_us, step_us)
    for i in list_of_times:
        fractions = [0.5, 1, (np.sin((i / 1e6) * (np.pi / 2)))**2, 1, 0.5]

        phases, sens = compute_phases_from_line_fit(
            pulse_train,
            fractions,
            pi_t=pi_t,
            integral_scale=integral_scale,
        )

        # Save sens as a proper list (you said you'll need this later)
        if sens_list is None:
            sens_list = list(np.array(sens, dtype=float))

        accm_phase.append(-phases[3] + phases[4])

    theory_phases = np.array(accm_phase) / np.pi
    return list_of_times, theory_phases, sens_list


In [70]:
list_of_max_times = []
list_of_sens = []
for trans_pair in list_of_pairs:
    times, phases, sens_list = compute_theory_phase_mod_and_sens(32000,trans_pair)
    # plt.figure()
    # plt.plot(times,phases)
    # plt.show()
    # print(sens_list[4]-sens_list[3], times[np.argmin(np.abs(2-phases))])
    list_of_max_times.append(times[np.argmin(np.abs(2-phases))])
    list_of_sens.append(sens_list[4]-sens_list[3])

In [78]:
print(np.array(list_of_max_times))

[ 1900  1900  3400  1900  2400 25600 10500  2900  2200  1900  3000  2400
  1600  1600  2000  3200  3100  2100  2000  2300  8500  5100  1700  2400
  2100]


In [5]:
list_of_max_times = [1900, 1900, 3400, 1900, 2400, 25600, 10500, 2900, 2200, 1900, 3000, 2400, 1600, 1600, 2000, 3200, 3100, 2100, 2000, 2300, 8500, 5100, 1700, 2400, 2100]

In [6]:
for i, j in zip(list_of_pairs, list_of_max_times):
    print(i, j)

[[0, 1, 1], [0, 2, 0]] 1900
[[0, 2, 2], [0, 2, 0]] 1900
[[0, 3, 0], [0, 2, 0]] 3400
[[0, 3, 2], [0, 2, 0]] 1900
[[0, 3, 1], [0, 2, 0]] 2400
[[0, 3, -1], [0, 2, 0]] 25600
[[0, 4, 1], [0, 2, 0]] 10500
[[0, 4, 0], [0, 2, 0]] 2900
[[0, 4, -1], [0, 2, 0]] 2200
[[0, 4, -2], [0, 2, 0]] 1900
[[-2, 3, -2], [-2, 3, -1]] 3000
[[-2, 3, -3], [-2, 3, -1]] 2400
[[1, 3, 3], [1, 3, 2]] 1600
[[-1, 4, -3], [-1, 4, -2]] 1600
[[-2, 4, -4], [-2, 3, -3]] 2000
[[1, 4, 2], [1, 4, 1]] 3200
[[2, 4, 3], [2, 4, 2]] 3100
[[2, 4, 3], [2, 2, 2]] 2100
[[2, 4, 4], [2, 4, 3]] 2000
[[-2, 2, -2], [-2, 3, -3]] 2300
[[-2, 2, -1], [-2, 3, -2]] 8500
[[2, 2, 1], [2, 4, 2]] 5100
[[1, 3, 2], [1, 2, 2]] 1700
[[-2, 1, -1], [-2, 3, -3]] 2400
[[-2, 1, 0], [-2, 3, -1]] 2100


In [8]:
import ast
import numpy as np
from collections import defaultdict, deque

def _make_row_col_labels():
    Fs = [1, 2, 3, 4]
    states = []
    for F in Fs:
        for j in range(2 * F + 1):
            mF = F - j
            states.append([F, mF])  # row label = [F, mF] in D5/2
    row_labels = states                      # length 24
    col_labels = [-2, -1, 0, 1, 2]          # a = mF in S1/2 F=2 (your convention)
    return row_labels, col_labels

def _transition_to_rc(t, row_labels, col_labels):
    """
    t is [a, b, c] where:
      a -> column label
      b,c -> row label [F, mF]
    Returns (row_index, col_index) or (None, None) if not representable.
    """
    row_label = [t[1], t[2]]
    col_label = t[0]
    row_index = next((k for k, lbl in enumerate(row_labels) if lbl == row_label), None)
    col_index = col_labels.index(col_label) if col_label in col_labels else None
    return row_index, col_index

def build_detuning_matrix_from_file(
    filepath,
    anchor=(0, 2, 0),
    use_weights=True,
    weight_from_field_index=2,  # the "3rd entry" in your line (0-based in the tuple, after the pair)
    verbose=True,
):
    """
    Reads lines like:
      [[t1],[t2]], detuning, sigma?, other?, ['runs'...]
    and solves x(t1) - x(t2) = detuning, with x(anchor)=0.

    Returns:
      detuning_matrix_24x5 (np.ndarray),
      detuning_dict {transition_tuple: detuning_relative_to_anchor},
      diagnostics dict
    """
    row_labels, col_labels = _make_row_col_labels()

    # --- Parse file into records ---
    records = []
    with open(filepath, "r") as f:
        for ln, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                pair, det, *rest = ast.literal_eval(f"({line})")
                t1 = tuple(pair[0])
                t2 = tuple(pair[1])
                records.append((t1, t2, float(det), rest))
            except Exception as e:
                raise ValueError(f"Failed parsing line {ln}: {line}\nError: {e}")

    if len(records) == 0:
        raise ValueError("No valid records found.")

    # --- Build node list and adjacency (for reachability from anchor) ---
    adj = defaultdict(set)
    nodes = set()
    for t1, t2, det, rest in records:
        nodes.add(t1); nodes.add(t2)
        adj[t1].add(t2); adj[t2].add(t1)  # undirected for connectivity

    anchor = tuple(anchor)
    if anchor not in nodes:
        raise ValueError(f"Anchor transition {anchor} not found in file.")

    # Find connected component containing anchor
    reachable = set()
    q = deque([anchor])
    reachable.add(anchor)
    while q:
        u = q.popleft()
        for v in adj[u]:
            if v not in reachable:
                reachable.add(v)
                q.append(v)

    # Keep only equations where both ends are reachable
    eqs = [(t1, t2, det, rest) for (t1, t2, det, rest) in records
           if (t1 in reachable and t2 in reachable)]

    # --- Index transitions ---
    trans_list = sorted(reachable)
    idx = {t: i for i, t in enumerate(trans_list)}
    n = len(trans_list)
    m = len(eqs)

    # --- Build linear system A x = b for x_i - x_j = det ---
    A = np.zeros((m + 1, n), dtype=float)
    b = np.zeros(m + 1, dtype=float)
    w = np.ones(m + 1, dtype=float)

    for r, (t1, t2, det, rest) in enumerate(eqs):
        i = idx[t1]
        j = idx[t2]
        A[r, i] = 1.0
        A[r, j] = -1.0
        b[r] = det

        # Optional weighting
        if use_weights:
            # rest[0] corresponds to the 3rd item in the original line (after detuning),
            # but we allow selecting via weight_from_field_index for flexibility.
            # Original tuple is: (pair, detuning, field2, field3, runs)
            # We stored `rest` as [field2, field3, runs]
            # so field2 is rest[0].
            try:
                # map requested original index to rest index:
                # original indices: 0=pair,1=det,2=field2,3=field3,4=runs
                # rest index = original_index - 2
                ri = weight_from_field_index - 2
                sigma = float(rest[ri])
                if sigma > 0:
                    w[r] = 1.0 / (sigma ** 2)
            except Exception:
                pass

    # Anchor constraint: x(anchor)=0
    A[m, idx[anchor]] = 1.0
    b[m] = 0.0
    w[m] = 1e12  # very strong constraint

    # Weighted least squares: solve (sqrt(W)A)x = (sqrt(W)b)
    Wsqrt = np.sqrt(w)[:, None]
    Aw = A * Wsqrt
    bw = b * np.sqrt(w)

    x, residuals, rank, svals = np.linalg.lstsq(Aw, bw, rcond=None)

    # --- Diagnostics: how well equations are satisfied ---
    diffs = []
    for (t1, t2, det, rest) in eqs:
        pred = x[idx[t1]] - x[idx[t2]]
        diffs.append(pred - det)
    diffs = np.array(diffs)
    diagnostics = {
        "n_transitions_reachable": n,
        "n_equations_used": len(eqs),
        "rank": int(rank),
        "residual_rms": float(np.sqrt(np.mean(diffs**2))) if len(diffs) else np.nan,
        "residual_max_abs": float(np.max(np.abs(diffs))) if len(diffs) else np.nan,
        "n_total_records": len(records),
        "n_disconnected_transitions": len(nodes - reachable),
    }

    if verbose:
        print("Detuning solve diagnostics:")
        for k, v in diagnostics.items():
            print(f"  {k}: {v}")

    # --- Build detuning dict ---
    detuning_dict = {t: float(x[idx[t]]) for t in trans_list}

    # --- Fill 24x5 matrix ---
    detuning_matrix = np.full((24, 5), np.nan, dtype=float)
    for t, val in detuning_dict.items():
        r, c = _transition_to_rc(list(t), row_labels, col_labels)
        if r is not None and c is not None:
            detuning_matrix[r, c] = val

    return detuning_matrix, detuning_dict, diagnostics


# Example usage:
det_mat, det_map, diag = build_detuning_matrix_from_file("bussed_ramsey_detuning_calibration.txt", anchor=(0,2,0))
print(det_mat)


Detuning solve diagnostics:
  n_transitions_reachable: 11
  n_equations_used: 10
  rank: 11
  residual_rms: 3.0496999971400025e-14
  residual_max_abs: 5.684341886080802e-14
  n_total_records: 19
  n_disconnected_transitions: 14
[[            nan             nan -7.60778200e+01             nan
              nan]
 [            nan             nan             nan             nan
              nan]
 [            nan             nan             nan             nan
              nan]
 [            nan             nan -4.47861900e+01             nan
              nan]
 [            nan             nan             nan             nan
              nan]
 [            nan             nan  7.47808873e-15             nan
              nan]
 [            nan             nan             nan             nan
              nan]
 [            nan             nan             nan             nan
              nan]
 [            nan             nan             nan             nan
              nan]
 [     

In [9]:
import ast
import numpy as np
from collections import defaultdict, deque

def _make_row_col_labels():
    Fs = [1, 2, 3, 4]
    states = []
    for F in Fs:
        for j in range(2 * F + 1):
            mF = F - j
            states.append([F, mF])
    row_labels = states
    col_labels = [-2, -1, 0, 1, 2]
    return row_labels, col_labels

def _transition_to_rc(t, row_labels, col_labels):
    row_label = [t[1], t[2]]
    col_label = t[0]
    row_index = next((k for k, lbl in enumerate(row_labels) if lbl == row_label), None)
    col_index = col_labels.index(col_label) if col_label in col_labels else None
    return row_index, col_index

def build_detuning_via_bfs(filepath, anchor=(0, 2, 0), verbose=True):
    """
    Reads lines like:
      [[t1],[t2]], detuning, ..., ...
    and solves by graph propagation (BFS):
      x(t1) - x(t2) = detuning, with x(anchor)=0

    Returns:
      detuning_matrix_24x5,
      detuning_dict,
      diagnostics
    """
    anchor = tuple(anchor)
    row_labels, col_labels = _make_row_col_labels()

    # --- Parse records ---
    records = []
    with open(filepath, "r") as f:
        for ln, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                pair, det, *rest = ast.literal_eval(f"({line})")
                t1 = tuple(pair[0])
                t2 = tuple(pair[1])
                records.append((t1, t2, float(det)))
            except Exception as e:
                raise ValueError(f"Failed parsing line {ln}: {line}\nError: {e}")

    if anchor not in {t for rec in records for t in rec[:2]}:
        raise ValueError(f"Anchor {anchor} not found in file.")

    # --- Build directed adjacency with propagation rules ---
    # For equation x(t1) - x(t2) = det:
    #   x(t1) = x(t2) + det  -> edge t2 -> t1 with +det
    #   x(t2) = x(t1) - det  -> edge t1 -> t2 with -det
    adj = defaultdict(list)
    all_nodes = set()
    for t1, t2, det in records:
        all_nodes.add(t1); all_nodes.add(t2)
        adj[t2].append((t1, +det))
        adj[t1].append((t2, -det))

    # --- BFS propagate from anchor ---
    x = {anchor: 0.0}
    q = deque([anchor])

    # Track inconsistencies if the graph has loops / conflicting measurements
    inconsistencies = []  # (node, old, new, diff, from_node)

    while q:
        u = q.popleft()
        for v, delta in adj[u]:
            proposed = x[u] + delta
            if v not in x:
                x[v] = proposed
                q.append(v)
            else:
                # if we already have a value, check consistency
                diff = proposed - x[v]
                if abs(diff) > 1e-9:
                    inconsistencies.append((v, x[v], proposed, diff, u))

    # --- Fill 24x5 matrix with whatever is representable ---
    detuning_matrix = np.full((24, 5), np.nan, dtype=float)
    mapped = 0
    unmapped_but_known = []

    for t, val in x.items():
        r, c = _transition_to_rc(list(t), row_labels, col_labels)
        if r is not None and c is not None:
            detuning_matrix[r, c] = val
            mapped += 1
        else:
            unmapped_but_known.append(t)

    diagnostics = {
        "n_records": len(records),
        "n_nodes_total": len(all_nodes),
        "n_nodes_reachable_from_anchor": len(x),
        "n_matrix_entries_filled": mapped,
        "n_known_but_not_in_24x5_grid": len(unmapped_but_known),
        "n_inconsistency_flags": len(inconsistencies),
    }

    if verbose:
        print("BFS detuning diagnostics:")
        for k, v in diagnostics.items():
            print(f"  {k}: {v}")

        if inconsistencies:
            print("\nExample inconsistencies (showing up to 10):")
            for (node, old, new, diff, frm) in inconsistencies[:10]:
                print(f"  node={node} old={old:.6g} new={new:.6g} diff={diff:.6g} via={frm}")

        if unmapped_but_known:
            print("\nKnown transitions that don't fit your 24x5 labels (showing up to 10):")
            for t in unmapped_but_known[:10]:
                print(f"  {t}")

    return detuning_matrix, x, diagnostics
det_mat, det_map, diag = build_detuning_via_bfs("bussed_ramsey_detuning_calibration.txt", anchor=(0,2,0))
print(det_mat)

BFS detuning diagnostics:
  n_records: 19
  n_nodes_total: 25
  n_nodes_reachable_from_anchor: 11
  n_matrix_entries_filled: 11
  n_known_but_not_in_24x5_grid: 0
  n_inconsistency_flags: 0
[[       nan        nan -76.07782         nan        nan]
 [       nan        nan        nan        nan        nan]
 [       nan        nan        nan        nan        nan]
 [       nan        nan -44.78619         nan        nan]
 [       nan        nan        nan        nan        nan]
 [       nan        nan   0.              nan        nan]
 [       nan        nan        nan        nan        nan]
 [       nan        nan        nan        nan        nan]
 [       nan        nan        nan        nan        nan]
 [       nan        nan -70.66661         nan        nan]
 [       nan        nan -32.87493         nan        nan]
 [       nan        nan   5.247815        nan        nan]
 [       nan        nan  -1.086446        nan        nan]
 [       nan        nan        nan        nan        nan]

In [11]:
import ast
import numpy as np
from collections import defaultdict, deque

def make_D_row_labels():
    # 24 D5/2 hyperfine rows in your lab ordering
    Fs = [1, 2, 3, 4]
    rows = []
    for F in Fs:
        for j in range(2 * F + 1):
            mF = F - j
            rows.append((F, mF))
    return rows  # length 24

def build_D_detunings_via_bfs(filepath, anchor_D=(2, 0), require_same_a=True, verbose=True):
    """
    Interpret each line as: x(D1) - x(D2) = detuning
    where D = (b,c) from transition [a,b,c].

    anchor_D sets x(anchor_D)=0.
    If require_same_a=True, only use constraints where the two transitions share the same 'a'.
    """
    anchor_D = tuple(anchor_D)
    D_rows = make_D_row_labels()

    # Parse
    records = []
    with open(filepath, "r") as f:
        for ln, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                pair, det, *rest = ast.literal_eval(f"({line})")
                t1 = pair[0]  # [a,b,c]
                t2 = pair[1]
                records.append((t1, t2, float(det), ln))
            except Exception as e:
                raise ValueError(f"Failed parsing line {ln}:\n{line}\nError: {e}")

    # Build adjacency on D-level nodes
    # For x(D1) - x(D2) = det:
    #   x(D1) = x(D2) + det  (edge D2 -> D1 with +det)
    #   x(D2) = x(D1) - det  (edge D1 -> D2 with -det)
    adj = defaultdict(list)
    all_nodes = set()
    skipped_diff_a = 0

    for t1, t2, det, ln in records:
        a1, b1, c1 = t1
        a2, b2, c2 = t2

        if require_same_a and (a1 != a2):
            skipped_diff_a += 1
            continue

        D1 = (b1, c1)
        D2 = (b2, c2)

        all_nodes.add(D1); all_nodes.add(D2)
        adj[D2].append((D1, +det, ln))
        adj[D1].append((D2, -det, ln))

    if anchor_D not in all_nodes:
        raise ValueError(f"Anchor D-level {anchor_D} not present in usable constraints.")

    # BFS propagation
    x = {anchor_D: 0.0}
    q = deque([anchor_D])

    inconsistencies = []  # (node, old, new, diff, via_node, line_number)

    while q:
        u = q.popleft()
        for v, delta, ln in adj[u]:
            proposed = x[u] + delta
            if v not in x:
                x[v] = proposed
                q.append(v)
            else:
                diff = proposed - x[v]
                if abs(diff) > 1e-9:
                    inconsistencies.append((v, x[v], proposed, diff, u, ln))

    # Build a 24x1 "D detuning table" in your row order
    det_vec = np.full((24,), np.nan, dtype=float)
    for i, D in enumerate(D_rows):
        if D in x:
            det_vec[i] = x[D]

    diagnostics = {
        "n_records_total": len(records),
        "require_same_a": require_same_a,
        "n_skipped_diff_a": skipped_diff_a,
        "n_D_nodes_total_in_graph": len(all_nodes),
        "n_D_nodes_reachable_from_anchor": len(x),
        "n_D_rows_filled": int(np.sum(~np.isnan(det_vec))),
        "n_inconsistency_flags": len(inconsistencies),
    }

    if verbose:
        print("D-level BFS diagnostics:")
        for k, v in diagnostics.items():
            print(f"  {k}: {v}")
        if inconsistencies:
            print("\nExample inconsistencies (up to 10):")
            for node, old, new, diff, via, ln in inconsistencies[:10]:
                print(f"  line {ln}: node={node} old={old:.6g} new={new:.6g} diff={diff:.6g} via={via}")

    return det_vec, x, diagnostics, D_rows

# Example:
det_vec, det_map, diag, D_rows = build_D_detunings_via_bfs("bussed_ramsey_detuning_calibration.txt", anchor_D=(2,0))
print(det_vec)  # 24-length array, matching your D-row ordering


D-level BFS diagnostics:
  n_records_total: 19
  require_same_a: True
  n_skipped_diff_a: 0
  n_D_nodes_total_in_graph: 20
  n_D_nodes_reachable_from_anchor: 20
  n_D_rows_filled: 20
  n_inconsistency_flags: 0
[-76.07782           nan         nan -44.78619           nan   0.
         nan  35.090992  -64.760708  -70.66661   -32.87493     5.247815
  -1.086446   -0.4829299  30.052084  -40.942006  -45.652721  -19.655921
 -14.60726     7.951547    6.456089    9.554785   16.892773   32.111571 ]


In [41]:
import ast
import numpy as np
from collections import defaultdict

def make_row_col_labels():
    Fs = [1, 2, 3, 4]
    states = []
    for F in Fs:
        for j in range(2 * F + 1):
            mF = F - j
            states.append([F, mF])  # D5/2 [F, mF]
    row_labels = states                      # length 24
    col_labels = [-2, -1, 0, 1, 2]          # S1/2 a
    return row_labels, col_labels

def transition_to_rc(t, row_labels, col_labels):
    # t = [a,b,c]
    row_label = [t[1], t[2]]
    col_label = t[0]
    row_index = next((k for k, lbl in enumerate(row_labels) if lbl == row_label), None)
    col_index = col_labels.index(col_label) if col_label in col_labels else None
    return row_index, col_index

def parse_detuning_file(filepath):
    """
    Parses lines like:
      [[t1],[t2]], detuning, ...
    Returns list of (t1_tuple, t2_tuple, detuning_float).
    """
    records = []
    with open(filepath, "r") as f:
        for ln, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                pair, det, *rest = ast.literal_eval(f"({line})")
                t1 = tuple(pair[0])  # (a,b,c)
                t2 = tuple(pair[1])
                records.append((t1, t2, float(det)))
            except Exception as e:
                raise ValueError(f"Failed parsing line {ln}:\n{line}\nError: {e}")
    return records

def build_transition_graph(records, require_same_a=True, link_same_D_across_a=True):
    """
    Each record encodes: x(t1) - x(t2) = det
    We create directed edges u->v with delta such that x(v) = x(u) + delta.

    So:
      x(t1) = x(t2) + det  => edge t2 -> t1 with +det
      x(t2) = x(t1) - det  => edge t1 -> t2 with -det

    Additionally, if link_same_D_across_a=True, add 0 edges between transitions
    that share the same D-level (b,c).
    """
    adj = defaultdict(list)     # u -> list of (v, delta)
    nodes = set()
    skipped_diff_a = 0

    # 1) edges from measured constraints
    for t1, t2, det in records:
        a1, b1, c1 = t1
        a2, b2, c2 = t2
        if require_same_a and (a1 != a2):
            skipped_diff_a += 1
            continue
        nodes.add(t1); nodes.add(t2)
        adj[t2].append((t1, +det))
        adj[t1].append((t2, -det))

    # 2) 0-edges between same D-level (b,c), across different a
    if link_same_D_across_a:
        by_D = defaultdict(list)
        for t in nodes:
            a,b,c = t
            by_D[(b,c)].append(t)
        for D, ts in by_D.items():
            # fully connect within this D group with 0 deltas
            for i in range(len(ts)):
                for j in range(i+1, len(ts)):
                    u, v = ts[i], ts[j]
                    adj[u].append((v, 0.0))
                    adj[v].append((u, 0.0))

    return adj, nodes, skipped_diff_a

def find_up_to_k_paths_with_values(adj, anchor, target, k=2, max_depth=10, tol=1e-9):
    """
    Find up to k distinct detuning values for target relative to anchor by
    exploring paths (DFS) from anchor to target.

    Returns list of (value, path_nodes_list) where path_nodes_list is [anchor,...,target].
    """
    results = []
    seen_vals = []

    # stack entries: (current_node, accumulated_value, path_list)
    stack = [(anchor, 0.0, [anchor])]
    visited_in_path = set([anchor])

    while stack and len(results) < k:
        cur, val, path = stack.pop()
        if len(path) > max_depth:
            continue

        if cur == target:
            # keep only distinct values (within tol)
            if not any(abs(val - sv) <= tol for sv in seen_vals):
                seen_vals.append(val)
                results.append((val, path))
            continue

        # explore neighbors, avoiding cycles in the current path
        path_set = set(path)
        for nxt, delta in adj.get(cur, []):
            if nxt in path_set:
                continue
            stack.append((nxt, val + delta, path + [nxt]))

    return results

def format_chain(path_nodes, value):
    """
    path_nodes is [anchor,...,target]. User wants: [target]-[...]-[anchor] = value
    """
    rev = list(reversed(path_nodes))  # [target,...,anchor]
    chain = "-".join([str(list(n)) for n in rev])
    return f"{chain} = {value}"

def build_detuning_matrix_and_mappings(filepath,anchor=(0,2,0),require_same_a=True,link_same_D_across_a=True,k_paths=5,max_depth=1):

    row_labels, col_labels = make_row_col_labels()
    records = parse_detuning_file(filepath)
    adj, nodes, skipped = build_transition_graph(
        records,
        require_same_a=require_same_a,
        link_same_D_across_a=link_same_D_across_a
    )

    anchor = tuple(anchor)
    if anchor not in nodes:
        raise ValueError(f"Anchor transition {anchor} not found among usable nodes. "
                         f"(Maybe require_same_a skipped it?)")

    detuning_matrix = np.full((24,5), np.nan, dtype=object)
    mapping = {}

    filled = 0
    reachable = 0

    for r, (F, mF) in enumerate(row_labels):
        for c, a in enumerate(col_labels):
            t = (a, F, mF)
            if t not in nodes:
                continue  
            paths = find_up_to_k_paths_with_values(
                adj, anchor, t, k=k_paths, max_depth=max_depth
            )
            if not paths:
                continue
            reachable += 1
            vals = [v for (v, p) in paths]
            chains = [format_chain(p, v) for (v, p) in paths]
            mapping[t] = chains

            if len(vals) == 1:
                detuning_matrix[r, c] = vals[0]
            else:
                detuning_matrix[r, c] = tuple(vals)
            filled += 1

    diagnostics = {
        "n_records": len(records),
        "n_nodes_in_graph": len(nodes),
        "n_skipped_diff_a": skipped,
        "n_matrix_cells_filled": filled,
        "n_transitions_reachable_within_grid": reachable,
        "require_same_a": require_same_a,
        "link_same_D_across_a": link_same_D_across_a,
        "k_paths": k_paths,
        "max_depth": max_depth
    }

    return detuning_matrix, mapping, diagnostics


# Example usage:
det_mat, mapping, diag = build_detuning_matrix_and_mappings(
    "bussed_ramsey_detuning_calibration.txt",
    anchor=(0,2,0),
    require_same_a=True,
    link_same_D_across_a=True,
    k_paths=5,
    max_depth=12
)
print(diag)
print(det_mat)


{'n_records': 24, 'n_nodes_in_graph': 30, 'n_skipped_diff_a': 0, 'n_matrix_cells_filled': 30, 'n_transitions_reachable_within_grid': 30, 'require_same_a': True, 'link_same_D_across_a': True, 'k_paths': 5, 'max_depth': 12}
[[nan nan -76.07782 nan nan]
 [21.910034 nan nan nan nan]
 [22.935996 nan nan nan nan]
 [nan nan (-100.78296, -44.78619) (-100.78296, -44.78619) nan]
 [nan nan nan nan -28.411329]
 [nan nan 0.0 nan nan]
 [-2.5002899000000003 nan nan nan nan]
 [32.206488 nan nan nan nan]
 [nan nan nan (-64.76070800000001, -8.763937999999996) nan]
 [nan nan (-70.66661, -14.669839999999997)
  (-70.66661, -14.669839999999997) nan]
 [nan nan -32.87493 nan nan]
 [nan nan 5.247815 nan nan]
 [-1.086446 nan -1.086446 nan nan]
 [-0.4829299 nan nan nan nan]
 [30.052084 nan nan nan nan]
 [nan nan nan nan -40.942006]
 [nan nan nan nan -45.652721]
 [nan nan nan -19.655921 -19.655921]
 [nan nan -14.60726 -14.60726 nan]
 [nan nan 7.951547 nan nan]
 [nan nan 6.456089 nan nan]
 [nan 8.100645 8.100645 n

In [135]:
def print_mappings_for_all_transitions_in_file(filepath, mapping):
    records = parse_detuning_file(filepath)

    unique_ts = sorted({t for (t1, t2, det) in records for t in (t1, t2)})

    for t in unique_ts:
        chains = mapping.get(t, [])
        print(f"\nMappings for {list(t)}:")
        if not chains:
            print("no path to anchor found")
        else:
            for s in chains:
                print("  ", s)

print_mappings_for_all_transitions_in_file("bussed_ramsey_detuning_calibration.txt", mapping)



Mappings for [-2, 1, -1]:
   [-2, 1, -1]-[-2, 3, -3]-[-2, 3, -1]-[0, 3, -1]-[0, 2, 0] = 16.117266

Mappings for [-2, 1, 0]:
   [-2, 1, 0]-[-2, 3, -1]-[0, 3, -1]-[0, 2, 0] = 14.103086000000001

Mappings for [-2, 2, -2]:
   [-2, 2, -2]-[-2, 3, -3]-[-2, 3, -1]-[0, 3, -1]-[0, 2, 0] = 34.477044

Mappings for [-2, 2, -1]:
   [-2, 2, -1]-[-2, 3, -2]-[-2, 3, -1]-[0, 3, -1]-[0, 2, 0] = -33.925279

Mappings for [-2, 3, -3]:
   [-2, 3, -3]-[-2, 3, -1]-[0, 3, -1]-[0, 2, 0] = 30.228736

Mappings for [-2, 3, -2]:
   [-2, 3, -2]-[-2, 3, -1]-[0, 3, -1]-[0, 2, 0] = -29.906869

Mappings for [-2, 3, -1]:
   [-2, 3, -1]-[0, 3, -1]-[0, 2, 0] = -31.105934

Mappings for [-2, 4, -4]:
   [-2, 4, -4]-[-2, 3, -3]-[-2, 3, -1]-[0, 3, -1]-[0, 2, 0] = 34.25441

Mappings for [-1, 3, -1]:
   [-1, 3, -1]-[0, 3, -1]-[0, 2, 0] = -21.352724

Mappings for [-1, 4, -3]:
no path to anchor found

Mappings for [-1, 4, -2]:
no path to anchor found

Mappings for [0, 1, 1]:
   [0, 1, 1]-[0, 2, 0] = -146.6905

Mappings for [0, 2, 

In [136]:
import ast
import numpy as np
from collections import defaultdict

def make_row_col_labels():
    Fs = [1, 2, 3, 4]
    states = []
    for F in Fs:
        for j in range(2 * F + 1):
            mF = F - j
            states.append([F, mF])  # D5/2 [F, mF]
    row_labels = states                      # length 24
    col_labels = [-2, -1, 0, 1, 2]          # S1/2 a
    return row_labels, col_labels

def transition_to_rc(t, row_labels, col_labels):
    # t = [a,b,c]
    row_label = [t[1], t[2]]
    col_label = t[0]
    row_index = next((k for k, lbl in enumerate(row_labels) if lbl == row_label), None)
    col_index = col_labels.index(col_label) if col_label in col_labels else None
    return row_index, col_index

def parse_detuning_file(filepath):
    """
    Parses lines like:
      [[t1],[t2]], detuning, ...
    Returns list of (t1_tuple, t2_tuple, detuning_float).
    """
    records = []
    with open(filepath, "r") as f:
        for ln, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                pair, det, *rest = ast.literal_eval(f"({line})")
                t1 = tuple(pair[0])  # (a,b,c)
                t2 = tuple(pair[1])
                records.append((t1, t2, float(det)))
            except Exception as e:
                raise ValueError(f"Failed parsing line {ln}:\n{line}\nError: {e}")
    return records

def build_transition_graph(records, require_same_a=True, link_same_D_across_a=True):
    """
    File rule (reference-zero): x(t1) = det - x(t2)
    So traversing along a measurement edge uses: x_next = det - x_cur  (direction doesn't matter).

    For same-D links across 'a': enforce equality x_next = x_cur.
    """
    adj = defaultdict(list)  # u -> list of (v, kind, det)
    nodes = set()
    skipped_diff_a = 0

    # measurement edges
    for t1, t2, det in records:
        a1, b1, c1 = t1
        a2, b2, c2 = t2
        if require_same_a and (a1 != a2):
            skipped_diff_a += 1
            continue

        nodes.add(t1); nodes.add(t2)

        # undirected reflect edge with parameter det
        adj[t1].append((t2, "reflect", float(det)))
        adj[t2].append((t1, "reflect", float(det)))

    # equality edges across same D-level (b,c)
    if link_same_D_across_a:
        by_D = defaultdict(list)
        for t in nodes:
            a, b, c = t
            by_D[(b, c)].append(t)

        for ts in by_D.values():
            for i in range(len(ts)):
                for j in range(i + 1, len(ts)):
                    u, v = ts[i], ts[j]
                    adj[u].append((v, "equal", 0.0))
                    adj[v].append((u, "equal", 0.0))

    return adj, nodes, skipped_diff_a


def find_up_to_k_paths_with_values(adj, anchor, target, k=5, max_depth=10, tol=1e-9):
    """
    Traversal rule:
      - reflect edge with det: x_next = det - x_cur
      - equal edge:            x_next = x_cur
    """
    results = []
    seen_vals = []

    stack = [(anchor, 0.0, [anchor])]  # (node, x_value, path)

    while stack and len(results) < k:
        cur, xcur, path = stack.pop()
        if len(path) > max_depth:
            continue

        if cur == target:
            if not any(abs(xcur - sv) <= tol for sv in seen_vals):
                seen_vals.append(xcur)
                results.append((xcur, path))
            continue

        path_set = set(path)
        for nxt, kind, det in adj.get(cur, []):
            if nxt in path_set:
                continue

            if kind == "equal":
                xnext = xcur
            else:  # "reflect"
                xnext = det + xcur

            stack.append((nxt, xnext, path + [nxt]))

    return results


def format_chain(path_nodes, value):
    """
    path_nodes is [anchor,...,target]. User wants: [target]-[...]-[anchor] = value
    """
    rev = list(reversed(path_nodes))  # [target,...,anchor]
    chain = "-".join([str(list(n)) for n in rev])
    return f"{chain} = {value}"

def build_detuning_matrix_and_mappings(filepath,anchor=(0,2,0),require_same_a=True,link_same_D_across_a=True,k_paths=5,max_depth=1):

    row_labels, col_labels = make_row_col_labels()
    records = parse_detuning_file(filepath)
    adj, nodes, skipped = build_transition_graph(
        records,
        require_same_a=require_same_a,
        link_same_D_across_a=link_same_D_across_a
    )

    anchor = tuple(anchor)
    if anchor not in nodes:
        raise ValueError(f"Anchor transition {anchor} not found among usable nodes. "
                         f"(Maybe require_same_a skipped it?)")

    detuning_matrix = np.full((24,5), np.nan, dtype=object)
    mapping = {}

    filled = 0
    reachable = 0

    for r, (F, mF) in enumerate(row_labels):
        for c, a in enumerate(col_labels):
            t = (a, F, mF)
            if t not in nodes:
                continue  
            paths = find_up_to_k_paths_with_values(
                adj, anchor, t, k=k_paths, max_depth=max_depth
            )
            if not paths:
                continue
            reachable += 1
            vals = [v for (v, p) in paths]
            chains = [format_chain(p, v) for (v, p) in paths]
            mapping[t] = chains

            if len(vals) == 1:
                detuning_matrix[r, c] = vals[0]
            else:
                detuning_matrix[r, c] = tuple(vals)
            filled += 1

    diagnostics = {
        "n_records": len(records),
        "n_nodes_in_graph": len(nodes),
        "n_skipped_diff_a": skipped,
        "n_matrix_cells_filled": filled,
        "n_transitions_reachable_within_grid": reachable,
        "require_same_a": require_same_a,
        "link_same_D_across_a": link_same_D_across_a,
        "k_paths": k_paths,
        "max_depth": max_depth
    }

    return detuning_matrix, mapping, diagnostics


# Example usage:
det_mat, mapping, diag = build_detuning_matrix_and_mappings(
    "bussed_ramsey_detuning_calibration.txt",
    anchor=(0,2,0),
    require_same_a=False,
    link_same_D_across_a=False,
    k_paths=8,
    max_depth=12
)
print(diag)
print(np.array(det_mat, dtype = float))


{'n_records': 30, 'n_nodes_in_graph': 32, 'n_skipped_diff_a': 0, 'n_matrix_cells_filled': 30, 'n_transitions_reachable_within_grid': 30, 'require_same_a': False, 'link_same_D_across_a': False, 'k_paths': 8, 'max_depth': 12}
[[        nan         nan -146.6905           nan         nan]
 [  14.103086         nan         nan         nan         nan]
 [  16.117266         nan         nan         nan         nan]
 [        nan         nan  -86.08038    -1.38569   -26.712168]
 [        nan         nan         nan         nan   10.09285 ]
 [        nan         nan    0.               nan         nan]
 [ -33.925279         nan         nan         nan         nan]
 [  34.477044         nan         nan         nan         nan]
 [        nan         nan         nan  -48.44186          nan]
 [        nan         nan -135.7951    -60.01033          nan]
 [        nan         nan  -63.67267          nan         nan]
 [        nan         nan   10.2299           nan         nan]
 [ -31.105934  -21.3

In [134]:
import numpy as np
f0 = 546.0996195796035
f1 = 623.890150051562
# Specify the path and filename
filename_an1 = r'Z:\Lab Data\D52_Calibration_Ba137\Calibration_Parameters\an1_2026.txt'
filename_an2 = r'Z:\Lab Data\D52_Calibration_Ba137\Calibration_Parameters\an2_2026.txt'
def read_mat(filename):
    with open(filename, 'r') as file:
        lines = file.readlines()
    matrix = []
    for line in lines:
        row = [float(x) for x in line.split()]
        matrix.append(row)
    return np.array(matrix)
matrix =  np.array([[         np.nan, 457.74885809, 460.70069237, 463.65252665,
        466.60216223],
       [466.96523442, 469.92057413, 472.87591385, 475.83125357,
        478.78659329],
       [479.49650835, 482.45184807, 485.40592233, 488.35999659,
                 np.nan],
       [         np.nan,          np.nan, 531.90854739, 534.86038883,
        537.81045731],
       [         np.nan, 535.71541587, 538.66505145, 541.61468703,
        544.56432261],
       [539.37243953, 542.32856506, 545.282623  , 548.23445728,
        551.18629156],
       [545.79903465, 548.75437437, 551.70971409, 554.6650538 ,
                 np.nan],
       [551.93251007, 554.88784978, 557.8431895 ,          np.nan,
                 np.nan],
       [         np.nan,          np.nan,          np.nan, 593.57925724,
        596.52889282],
       [         np.nan,          np.nan, 594.63100532, 597.58214852,
        600.5317841 ],
       [         np.nan, 596.04519024, 598.9992645 , 601.95109877,
        604.90293305],
       [597.70461366, 600.65995337, 603.61408859, 606.56592286,
        609.51775714],
       [602.54187381, 605.49786766, 608.45194192, 611.40601618,
                 np.nan],
       [607.64452195, 610.59842172, 613.55249598,          np.nan,
                 np.nan],
       [613.1734413 , 616.12878102,          np.nan,          np.nan,
                 np.nan],
       [         np.nan,          np.nan,          np.nan,          np.nan,
        596.85291985],
       [         np.nan,          np.nan,          np.nan, 600.80649199,
        603.75612757],
       [         np.nan,          np.nan, 603.98153689, 606.63337117,
        609.58272399],
       [         np.nan, 606.05031215, 609.0043864 , 611.95622068,
        614.90805496],
       [608.02108391, 610.97515816, 613.92923242, 616.88150547,
        619.83099091],
       [612.56078504, 615.5148593 , 618.46893356, 621.42101298,
                 np.nan],
       [616.66311518, 619.6184549 , 622.57248451,          np.nan,
                 np.nan],
       [620.15168428, 623.107024  ,          np.nan,          np.nan,
                 np.nan],
       [620.45592666,          np.nan,          np.nan,          np.nan,
                 np.nan]])

# Read the text file and load the content into an array
matrix_an1 = read_mat(filename_an1)
matrix_an2 = read_mat(filename_an2)
# print(np.shape(matrix_an1),np.shape(matrix_an2))

# print(matrix_an1)
matrix_an2 = matrix_an2 + 1e-6*np.array(det_mat, dtype = float)
print(matrix_an2)


new_matrix = f0+matrix_an2+matrix_an1*(f1-f0)
# print(new_matrix)

differences = np.diff(new_matrix, axis=1)
# print(differences)
average_differences = np.nanmean(differences, axis=0)
print(average_differences)

def extrapolate_matrix_by_col_spacings(M, col_spacings, mode="mean"):

    M = np.array(M, dtype=float, copy=True)
    n_rows, n_cols = M.shape
    col_spacings = np.asarray(col_spacings, dtype=float)

    if col_spacings.shape[0] != n_cols - 1:
        raise ValueError(f"col_spacings must have length n_cols-1 = {n_cols-1}")

    offsets = np.zeros(n_cols, dtype=float)
    offsets[1:] = np.cumsum(col_spacings)

    def combine(preds):
        preds = np.asarray(preds, dtype=float)
        if mode == "median":
            return float(np.median(preds))
        return float(np.mean(preds))  

    filled_mask = np.zeros_like(M, dtype=bool)

    for r in range(n_rows):
        row = M[r]
        known_idx = np.where(~np.isnan(row))[0]
        if known_idx.size == 0:
            continue  

        missing_idx = np.where(np.isnan(row))[0]
        if missing_idx.size == 0:
            continue

        known_vals = row[known_idx]

        for j in missing_idx:
            preds = []
            for i, v_i in zip(known_idx, known_vals):
                # v_j ≈ v_i + (offset[j] - offset[i])
                preds.append(v_i + (offsets[j] - offsets[i]))
            row[j] = combine(preds)
            filled_mask[r, j] = True

        M[r] = row

    return M

filled_matrix = extrapolate_matrix_by_col_spacings(new_matrix, average_differences)
nan_positions = np.isnan(matrix)
filled_matrix[nan_positions] = np.nan
cali_freqs = filled_matrix
print(cali_freqs)

[[         nan          nan -14.11267928          nan          nan]
 [-39.48352956          nan          nan          nan          nan]
 [-93.44307407          nan          nan          nan          nan]
 [         nan          nan  59.06314135  46.43668872  33.83070592]
 [         nan          nan          nan          nan   3.82509474]
 [         nan          nan   0.                  nan          nan]
 [ -2.02031996          nan          nan          nan          nan]
 [-26.10455445          nan          nan          nan          nan]
 [         nan          nan          nan 126.05215968          nan]
 [         nan          nan 120.54783825 107.92131216          nan]
 [         nan          nan 101.27449964          nan          nan]
 [         nan          nan  81.38184895          nan          nan]
 [ 86.32378493  73.65581301  61.00860458          nan          nan]
 [ 65.32349772          nan          nan          nan          nan]
 [ 42.95623692          nan          nan        

In [92]:
((120.54783825 - 107.92131216) - (59.06314135 -46.43668872)) + ((59.06322743  -46.4366901) -(120.54797405 - 107.92137217))

8.90999999825226e-06

In [79]:
(59.06322743  -46.4366901) -(120.54797405 - 107.92137217)

-6.454999999760958e-05

In [80]:
(120.54797405 - 107.92137217) - (120.54783825 - 107.92131216)

7.578999999680036e-05

In [75]:
59.06314135 -46.43668872

12.626452630000003

In [82]:
2.94713401- 2.9470589 

7.511000000004486e-05

In [114]:
diffa = [2.94944356, 2.94729444, 2.94508564, 2.94293208]
diffb = [2.94944356, 2.94729434, 2.94508564, 2.94293208]
print(np.array(diffb) - np.array(diffa))

[ 0.00000000e+00 -9.99999998e-08  0.00000000e+00  0.00000000e+00]


In [129]:
def Get_1762_EOM_Freqs_an1an2(want_array,f_offset,f_upper):
    #Input definition:
    #want_array: a list of sets of 3 numbers e.g. [[2,2,0],[-1,4,-4]], which defines 
    #the desired S to D EOM transition frequency. The first number denotes the 
    #m number in the S level, the second number denotes the F number in the D level,
    #the third number denotes the m number in the D level.
    #f_offset: scalar number of the offset transition EOM frequency used for the an1 and an2 estimation in MHz.
    #f_upper: scalar number of the upper transition EOM frequency used for the an1 and an2 estimation in MHz.
    #
    #Output definition:
    #Freqs: a 1D array of numbers, of the same length as the list in want_array.
    #Returns the estimated frequencies in MHz using an1 and an2 of the transitions dictated
    #in want_array.
    an1 = np.loadtxt("Z:\\Lab Data\\D52_Calibration_Ba137\\Calibration_Parameters\\an1_2026.txt")
    an2 = np.loadtxt("Z:\\Lab Data\\D52_Calibration_Ba137\\Calibration_Parameters\\an2_2026.txt")
    m_S_num_list = np.asarray([-2,-1,0,1,2])
    F_num_list = np.asarray([1,1,1,2,2,2,2,2,3,3,3,3,3,3,3,4,4,4,4,4,4,4,4,4])
    m_D_num_list = np.asarray([1,0,-1,2,1,0,-1,-2,3,2,1,0,-1,-2,-3,4,3,2,1,0,-1,-2,-3,-4])
    Freqs_table = np.empty([np.size(an1,0),np.size(an1,1)])

    for D_index in range(np.size(an1,0)):
        for S_index in range(np.size(an1,1)):
            Freqs_table[D_index,S_index] = an1[D_index,S_index]*(f_upper - f_offset) + an2[D_index,S_index] + f_offset            
    
    Freqs_S_diff = np.nanmean(Freqs_table[:,1:] - Freqs_table[:,0:-1],0)
    print(Freqs_S_diff)
    S_Freq_Shifts = np.empty([np.size(Freqs_table,1),np.size(Freqs_table,1)])
    
    for S_index in range(np.size(an1,1)):
        for neg_index in range(S_index+1):
            if neg_index == S_index:
                S_Freq_Shifts[S_index,neg_index] = 0
            else:
                S_Freq_Shifts[S_index,neg_index] = np.sum(Freqs_S_diff[neg_index:S_index])
        for pos_index in range(S_index+1,np.size(an1,1)):
            if pos_index == S_index:
                S_Freq_Shifts[S_index,pos_index] = 0
            else:
                S_Freq_Shifts[S_index,pos_index] = -np.sum(Freqs_S_diff[S_index:pos_index])
                
    for D_index in range(np.size(an1,0)):
        for S_index in range(np.size(an1,1)):
            if np.isnan(Freqs_table[D_index,S_index]):
                Freqs_table[D_index,S_index] = np.nanmean(Freqs_table[D_index,:] + S_Freq_Shifts[S_index,:])
                
    Freqs = np.empty(np.size(want_array,0))
    ind_count = 0
    for want_index in want_array:
        m_S_num = want_index[0]
        F_num = want_index[1]
        m_D_num = want_index[2]
        if m_S_num == 0 and F_num == 0 and  m_D_num == 0:
            Freqs[ind_count] = 800
        else:
            Freqs[ind_count] = Freqs_table[np.where((F_num_list == F_num) & (m_D_num_list == m_D_num)),np.where(m_S_num_list == m_S_num)]
        #Freqs[ind_count] = an1[np.where((F_num_list == F_num) & (m_D_num_list == m_D_num)),np.where(m_S_num_list == m_S_num)]* \
        #(f_upper - f_offset) \
        #+ an2[np.where((F_num_list == F_num) & (m_D_num_list == m_D_num)),np.where(m_S_num_list == m_S_num)] \
        #+ f_offset
        ind_count += 1

            
    return Freqs 

In [130]:
Get_1762_EOM_Freqs_an1an2([[0,2,0]], f0, f1)
# [2.94944356 2.94729434 2.94508564 2.94293208]

[2.94944356 2.94729434 2.94508564 2.94293208]


C:\Users\iamga\AppData\Local\Temp\ipykernel_31824\174102740.py:55: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  Freqs[ind_count] = Freqs_table[np.where((F_num_list == F_num) & (m_D_num_list == m_D_num)),np.where(m_S_num_list == m_S_num)]


array([546.09961958])

In [101]:
f0 = 546.0996195796035

In [99]:
f0

-1.0603964710753644e-06

In [ ]:
import numpy as np

def extrapolate_matrix_by_col_spacings(M, col_spacings, mode="mean"):

    M = np.array(M, dtype=float, copy=True)
    n_rows, n_cols = M.shape
    col_spacings = np.asarray(col_spacings, dtype=float)

    if col_spacings.shape[0] != n_cols - 1:
        raise ValueError(f"col_spacings must have length n_cols-1 = {n_cols-1}")

    offsets = np.zeros(n_cols, dtype=float)
    offsets[1:] = np.cumsum(col_spacings)

    def combine(preds):
        preds = np.asarray(preds, dtype=float)
        if mode == "median":
            return float(np.median(preds))
        return float(np.mean(preds))  

    filled_mask = np.zeros_like(M, dtype=bool)

    for r in range(n_rows):
        row = M[r]
        known_idx = np.where(~np.isnan(row))[0]
        if known_idx.size == 0:
            continue  

        missing_idx = np.where(np.isnan(row))[0]
        if missing_idx.size == 0:
            continue

        known_vals = row[known_idx]

        for j in missing_idx:
            preds = []
            for i, v_i in zip(known_idx, known_vals):
                # v_j ≈ v_i + (offset[j] - offset[i])
                preds.append(v_i + (offsets[j] - offsets[i]))
            row[j] = combine(preds)
            filled_mask[r, j] = True

        M[r] = row

    return M


M_filled = extrapolate_matrix_by_col_spacings(M, col_spacings, mode="mean")

np.set_printoptions(precision=8, suppress=True)



Filled matrix:
 [[455.65054794 458.59995261 461.54731869 464.49237771 467.43530366]
 [467.79822115 470.74766471 473.69495915 476.64004479 479.58297687]
 [480.30074064 483.25020303 486.19748806 489.1425737  492.08550578]
 [526.86006259 529.80950615 532.75678803 535.70190081 538.64481628]
 [533.59829809 536.54774165 539.49503609 542.44012173 545.38305381]
 [540.20287024 543.15232561 546.09962064 549.04470628 551.98765136]
 [546.61476586 549.56420942 552.51150386 555.4565895  558.39952158]
 [552.73667792 555.68612148 558.63341592 561.57850156 564.52143364]
 [585.58685827 588.53630183 591.48359627 594.42868191 597.37161399]
 [589.58141082 592.53085438 595.47816752 598.42321575 601.36616654]
 [593.93970678 596.88915034 599.83644478 602.78153042 605.7244625 ]
 [598.5443737  601.49385154 604.44107743 607.38619735 610.32912943]
 [603.37159636 606.32104531 609.26832025 612.21341709 615.15634917]
 [608.46147009 611.4108776  614.35819007 617.30327571 620.24620779]
 [613.97962468 616.92906824 619.

Filled matrix:
 [[  0.         458.59995261 461.54731869 464.49237771   0.        ]
 [467.79822115   0.           0.           0.           0.        ]
 [480.30074064 483.25020303   0.           0.           0.        ]
 [  0.           0.         532.75678803 535.70190081 538.64481628]
 [  0.           0.           0.           0.         545.38305381]
 [540.20287024 543.15232561   0.           0.         551.98765136]
 [546.61476586   0.           0.           0.           0.        ]
 [552.73667792   0.           0.           0.           0.        ]
 [  0.           0.           0.         594.42868191   0.        ]
 [  0.           0.         595.47816752 598.42321575   0.        ]
 [  0.           0.         599.83644478   0.           0.        ]
 [  0.         601.49385154 604.44107743   0.           0.        ]
 [603.37159636 606.32104531 609.26832025   0.           0.        ]
 [608.46147009 611.4108776    0.           0.           0.        ]
 [613.97962468   0.           0.